# Extract Boundary Tokens - Full Text
Processes ENTIRE corpus (not just 512 tokens)

In [ ]:
from google.colab import drive
import os, time
drive.mount('/content/drive')
time.sleep(2)
os.chdir('/content/drive/MyDrive/khabar-segmentation')
print(f"Working directory: {os.getcwd()}")

In [ ]:
!pip install transformers torch tqdm -q
print("Dependencies OK")

In [ ]:
import json
import torch
import numpy as np
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForTokenClassification
from tqdm import tqdm

# Load model from Drive
model_path = Path('checkpoints/camelbert_binary_classification_final')
print(f"Loading model from {model_path}...")
tokenizer = AutoTokenizer.from_pretrained(str(model_path))
model = AutoModelForTokenClassification.from_pretrained(str(model_path))
model.eval()
if torch.cuda.is_available():
    model = model.cuda()
print("Model loaded")

In [ ]:
# Load corpus
corpus_file = Path('data/processed/kitab_uqala_reference_corpus.txt')
print(f"Loading corpus from {corpus_file}...")
with open(corpus_file, encoding='utf-8') as f:
    text = f.read()
print(f"Corpus: {len(text):,} chars")

In [ ]:
# Process FULL TEXT in chunks
print("Running inference on FULL CORPUS...")
print(f"Text length: {len(text):,} chars")

all_tokens = []
all_predictions = []
all_offsets = []

CHUNK_SIZE = 500  # Slightly less than max to avoid edge effects
chunks_processed = 0

# Split text into overlapping chunks
for start_char in tqdm(range(0, len(text), CHUNK_SIZE), desc="Processing chunks"):
    end_char = min(start_char + CHUNK_SIZE + 50, len(text))  # Small overlap
    chunk_text = text[start_char:end_char]
    
    # Encode chunk
    encoded = tokenizer(
        chunk_text,
        return_tensors='pt',
        return_offsets_mapping=True,
        truncation=False,  # NO truncation - process full chunk
        padding=False,
    )
    
    # Run inference on chunk
    with torch.no_grad():
        if torch.cuda.is_available():
            outputs = model(
                input_ids=encoded['input_ids'].cuda(),
                attention_mask=encoded['attention_mask'].cuda()
            )
        else:
            outputs = model(**encoded)
        logits = outputs.logits[0]
    
    # Get predictions
    preds = np.argmax(logits.cpu().numpy(), axis=-1)
    tokens = tokenizer.convert_ids_to_tokens(encoded['input_ids'][0])
    offsets = encoded['offset_mapping'][0].numpy()
    
    # Avoid duplicate tokens from overlap
    if chunks_processed > 0 and len(all_tokens) > 0:
        # Skip first 5 tokens of new chunk to avoid duplicates
        preds = preds[5:]
        tokens = tokens[5:]
        offsets = offsets[5:]
    
    all_tokens.extend(tokens)
    all_predictions.extend(preds.tolist())
    all_offsets.extend(offsets.tolist())
    
    chunks_processed += 1

print(f"\nProcessing complete")
print(f"  Chunks processed: {chunks_processed}")
print(f"  Total tokens: {len(all_tokens):,}")
print(f"  Boundary tokens: {sum(all_predictions):,}")
print(f"  Percentage: {100 * sum(all_predictions) / len(all_predictions):.2f}%")

In [ ]:
# Extract boundary tokens
print("Extracting boundary tokens...")
boundary_tokens = []
boundary_indices = []

for idx, (token, pred) in enumerate(zip(all_tokens, all_predictions)):
    if pred == 1:  # Boundary token
        boundary_tokens.append(token)
        boundary_indices.append(idx)

print(f"Extracted: {len(boundary_tokens):,} boundary tokens")

In [ ]:
# Preview
print("\nFirst 50 boundary tokens:")
for i, token in enumerate(boundary_tokens[:50], 1):
    print(f"{i:3d}. {token}")

In [ ]:
# Save to JSON
print("\nSaving to JSON...")
results = {
    'metadata': {
        'corpus': 'kitab_uqala_reference_corpus.txt',
        'corpus_size_chars': len(text),
        'corpus_size_tokens': len(all_tokens),
        'model': 'camelbert_binary_classification_final',
        'processing_method': 'Chunked processing with overlap',
        'chunk_size': CHUNK_SIZE,
        'chunks_processed': chunks_processed,
        'boundary_tokens_count': len(boundary_tokens),
        'boundary_percentage': round(100 * len(boundary_tokens) / len(all_tokens), 2),
    },
    'boundary_tokens': boundary_tokens,
    'boundary_indices': boundary_indices,
}

output_file = Path('results/camelbert_boundary_tokens_clean.json')
output_file.parent.mkdir(parents=True, exist_ok=True)
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

file_size = output_file.stat().st_size / 1024
print(f"Saved: {output_file}")
print(f"Size: {file_size:.1f} KB")
print(f"\nBoundary tokens extracted: {len(boundary_tokens):,}")
print(f"Done!")